# Loading and Accessing Data

This notebook demonstrates how to load datasets from manifests or archives and access different types of protein data. We'll explore the Pythonic API for working with sequences, structures, assays, and MSAs.

## Loading Datasets

There are two main ways to load a PG2 dataset:
1. From a manifest file (TOML)
2. From a dataset archive (ZIP)

Let's start by importing the necessary modules:

In [ ]:
import dataclasses
from pathlib import Path
from pg2_dataset import Dataset, Manifest

# Set up paths
manifest_path = Path("../example_data/neime_2019.toml")

### Method 1: Loading from Manifest

In [ ]:
# Load manifest first
try:
    manifest = Manifest.from_path(manifest_path)
    print(f"Loaded manifest: {manifest.name}")

    # Create dataset from manifest
    dataset = Dataset.from_manifest(manifest)
    print(f"\nDataset created successfully!")
    print(f"Dataset name: {dataset.name}")

except Exception as e:
    print(f"Error loading from manifest: {e}")
    print("This might be due to missing data files or incorrect paths.")

In [ ]:
# Dump the data into .pgdata file.
dataset.dump(path="../example_data/")

### Method 2: Loading from Archive

If you have a dataset archive (created in the previous notebook), you can load it directly:

In [ ]:
# Look for existing archives
archive_path = "../example_data/NEIME_2019.pgdata"

try:
    dataset = Dataset.from_path(archive_path)
except Exception as e:
    print(f"Error loading from archive: {e}")
    print(f"Did you create an archive in the previous tutorial?")
    raise e

# The `Dataset` is a Pydantic BaseModel that validates the data upon loading.
# A `BaseModel`` has a `.model_fields` attribute returning the fields of the class.
dataset.model_fields

In [ ]:
# The `Dataset`` attributes are Python-native dataclasses
dataclasses.fields(dataset.assays[0])

## Exploring Dataset Structure

Let's examine what's in our dataset:

In [ ]:
print(f"Dataset: {dataset.name}")
print(f"Description: {dataset.description}")
print("\nDataset contents:")
print(f"  - Sequences: {len(dataset.sequences)}")
print(f"  - Structures: {len(dataset.structures)}")
print(f"  - MSAs: {len(dataset.msas)}")
print(f"  - Assays: {len(dataset.assays)}")
print(f"  - Assay variables: {len(dataset.assay_variables)}")

## Accessing Assays

In [ ]:
# Access the assays
assays = dataset.assays

# Extract an specific assay
my_assay = assays[0]

# We can get a summary of the data encoded in this assay:
for field in dataclasses.fields(my_assay):
    print(f"Found a field:\n{field}\n------------")

In [ ]:
# Access specific attributes such as name
name = my_assay.name
print(f"Assay name: {name}")

# Or extract the records
records = my_assay.records
print(f"{name} contains {len(records)} records")
print(f"record 1: {records[0][0]}... \n with value {records[0][1]}")

In [ ]:
# Easy to obtain the sequences and targets for your ML application:
sequences, targets = zip(*records)

## Accessing Assay Variables

Assay variables describe the experimental setup:

In [ ]:
print(f"Number of assay variables: {len(dataset.assay_variables)}")

for i, variable in enumerate(dataset.assay_variables):
    print(f"\nVariable {i + 1}:")
    print(f"  - Name: {variable.name}")
    print(f"  - Description: {variable.description}")
    print(f"  - Unit: {variable.unit}")
    print(f"  - Value: {variable.value}")

## Accessing Structures

In [ ]:
# Access the structures
structures = dataset.structures

# Obtain a specific structure:
my_structure = structures[0]

# We can get a summary of the metadata encoded in this assay:
for field in dataclasses.fields(my_structure):
    print(f"Found a field:\n{field}\n------------")

When you access the structure, we return a biopython structure object. Biopython follows the so-called SMCRA (Structure/Model/Chain/Residue/Atom) architecture. Allowing you to access atoms from residues, residues from chains, chains from models and models from structures.

The following methods can be used to extract specific atom or residue data. See https://biopython.org/wiki/The_Biopython_Structural_Bioinformatics_FAQ for more information. The page contains helpful tips for accessing information from the structure object.

```python
atom.get_name()  # atom name (spaces stripped, e.g. 'CA')
atom.get_id()  # id (equals atom name)
atom.get_coord()  # atomic coordinates
atom.get_vector()  # atomic coordinates as Vector object
atom.get_bfactor()  # isotropic B factor
atom.get_occupancy()  # occupancy
atom.get_altloc()  # alternative location specifier
atom.get_sigatm()  # std. dev. of atomic parameters
atom.get_siguij()  # std. dev. of anisotropic B factor
atom.get_anisou()  # anisotropic B factor
atom.get_fullname()  # atom name (with spaces, e.g. '.CA.')
```

```python
residue.get_resname()  # return the residue name (eg. 'GLY')
residue.is_disordered()  # 1 if the residue has disordered atoms
residue.get_segid()  # return the SEGID
residue.has_id(name)  # test if a residue has a certain atom
```

In [ ]:
biopython_structure = my_structure.value


def print_model_info(structure):
    """Prints out simple information describing the

    Args:
        structure: biopython structure
    """
    for model in structure:
        print(model)
        for chain in model:
            print(chain)
            for residue in chain:
                print(residue.get_resname())
                for atom in residue:
                    print(atom.get_name())
                    return


print_model_info(biopython_structure)

## Accessing MSAs (Multiple Sequence Alignments)

MSAs provide evolutionary information through aligned sequences. Similarly to structures we return the biopython object to access the msa data.

In [ ]:
# Access the MSA data
msas = dataset.msas

my_msa = msas[0]
print(my_msa)

In [ ]:
# A biopython MSA object is a list of sequence records:

biopython_msa = my_msa.value
print(biopython_msa)

In [ ]:
# Accessing individual SeqRecords in the MSA:
print(biopython_msa[0])

# or
biopython_msa[0]

In [ ]:
# Allows us to obtain the SeqRecord:
biopython_msa[0].seq

## Accessing Sequences
We also record the reference sequence(s) of the dataset in the `[[ sequence ]]` section. This is helpful for engineering compared to a `wild-type` or `starting_sequence` and `engineered_sequence` from previous campaigns.

**It is important to note that the sequences in dataset.sequences are your reference sequences. The mutated sequences belong to an assay are what you most likely use for your ML model. See 

In [ ]:
# Access the list of reference sequences
sequences = dataset.sequences

# Obtain a specific sequence
my_sequence = sequences[0]

print(f"Sequence name: {my_sequence.name}")
print(f"Sequence description: {my_sequence.description}")
print(f"Sequence type: {my_sequence.type}")
print(f"Sequence: {my_sequence.value[0:20]}....")

## Summary

In this notebook, we've learned how to:

1. **Load datasets** from manifests and archives
2. **Access different data types**: sequences, structures, MSAs, and assays
3. **Access metadata** and variables

The PG2 Dataset package provides a powerful and flexible way to work with protein data in machine learning workflows. The standardized API makes it easy to switch between different datasets while maintaining consistent code structure.

## Next Steps

Now you're ready to:
- Use PG2 Dataset in your own ML projects
- Share standardized datasets with collaborators
- Take a look at `04_Data_Access_for_ML` to get started connecting the data to your machine learning models.
